# T-338 — Writer fine-tune (SciReviewGen → Qwen2.5-3B-Instruct)

**ResearchGPT**, local-model arm. Trains a **survey-writing student** on the
human-written literature-review chapters in SciReviewGen.

| | |
|---|---|
| Base | `Qwen/Qwen2.5-3B-Instruct` |
| Method | QLoRA — 4-bit NF4, LoRA r=16 |
| Data | `kejian/SciReviewGen` — 84,705 train rows, `reference` → `target` |
| Licence | **CC BY-NC 4.0** (Kasanishi et al., Findings of ACL 2023, arXiv:2305.15186). Academic use satisfies it; state it in the report regardless of what any mirror card claims |
| Hardware | Kaggle T4 16 GB |

---

## ⚠️ This reverses a recorded decision — read `TRAINING_SPEC.md §4`

The Writer was **evaluated for fine-tuning on 2026-08-25 and rejected**. The
rejection was explicitly *not* about the data:

> SciReviewGen's `reference → target` pairs **are** a valid Writer training set
> — that part of the idea was sound. It fails on deployment, not on data.

Three objections were recorded. Two still stand; one has changed.

| objection | status now |
|---|---|
| ~10 min per Writer call on CPU (i5-10400, no GPU) vs 3.1 s hosted | **changed.** A 3B at `q4_K_M` on a 5 GB card is roughly 20–40 s/call. Still an order of magnitude slower than hosted, but no longer a non-starter |
| Writer context is the largest in the pipeline — insights from up to 60 papers, **~20,000 tokens** | **stands, and it is the binding one.** See below |
| SciReviewGen's own fine-tuned generators reached only **~30% of chapters matching human quality** | **stands.** This is the realistic ceiling |

### The context problem is the real blocker

`num_ctx 8192` — the largest window that leaves room on a 5 GB card — holds
roughly **25 papers' insights, not 60**. So a locally-served Writer is not the
same component as the hosted one; it is a Writer for smaller reviews. Train it
knowing that, and report `max_papers` alongside any number it produces.

### And a claim you must not make

Fine-tuning the generator **weakens the project's central finding**, which is
that *a small off-the-shelf model plus claim-level verification produces
grounded output*. This notebook is future-work evidence for T-806, not a
replacement for the hosted Writer. If it is adopted, the thesis sentence in the
report has to change.

## 1 · Configuration

In [ ]:
BASE_MODEL   = "Qwen/Qwen2.5-3B-Instruct"

SEED         = 20260829
N_TRAIN      = 4000          # of 84,705. A T4 will not chew the full set in 12 h
N_EVAL       = 200

LORA_R       = 16
LORA_ALPHA   = 32

EPOCHS       = 1             # 4,000 long examples is already ~3 h on a T4
LR           = 1e-4
BATCH        = 1
GRAD_ACCUM   = 16
MAX_SEQ_LEN  = 4096          # NOT 8192: see the context note in the header

ADAPTER_DIR  = "/kaggle/working/writer-qwen3b"

## 2 · Dependencies

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    """Install only what is absent. A re-run costs seconds, not minutes."""
    try:
        importlib.import_module(import_name or pkg.split("==")[0].split(">")[0].replace("-", "_"))
        print(f"  present : {pkg}")
    except ImportError:
        print(f"  install : {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# transformers/torch/datasets ship with the Kaggle image; the PEFT stack does not.
for pkg, mod in [
    ("transformers>=4.44", "transformers"),
    ("datasets", "datasets"),
    ("accelerate", "accelerate"),
    ("peft", "peft"),
    ("bitsandbytes", "bitsandbytes"),
    ("trl", "trl"),
]:
    ensure(pkg, mod)

import torch
print()
print("torch      :", torch.__version__)
print("cuda       :", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("device     :", name, f"({vram:.1f} GB)")
    # T4 is sm_75: NO bf16. Every cell below uses fp16 for this reason.
    print("bf16 ok    :", torch.cuda.is_bf16_supported(), "  <- False on T4, as expected")

## 3 · Load SciReviewGen and rebuild it as a Writer task

The dataset has **two columns**, and both intended uses are recoverable by
parsing:

```
reference   survey_title <s> chapter <s> abstract <s> BIBnnn </s>   (repeated)
target      the human-written chapter, with inline BIBnnn markers
```

Two transformations are needed before it is a Writer training set:

1. **`reference` → the Writer's actual input shape.** The pipeline's Writer sees
   per-paper *insights*, not raw abstracts. The closest honest mapping is
   abstract-as-insight, and the gap is worth stating in the report: this trains
   the student on richer input than the pipeline gives it at inference.

2. **`BIBnnn` → `[n]`.** SciReviewGen marks citations `BIB003`; the pipeline
   emits `[3]`, and `writers.py` resolves `[n]` against the reference list. A
   student trained on `BIBnnn` would emit markers the renderer silently drops —
   a citation that vanishes at render time is worse than no citation, because
   the sentence keeps its claim and loses its support.

In [ ]:
import re, json, random
from datasets import load_dataset

random.seed(SEED)
raw = load_dataset("kejian/SciReviewGen", split="train", streaming=True)

_BLOCK_SEP, _FIELD_SEP = "</s>", "<s>"
_BIB_RE = re.compile(r"BIB(\d+)")

def build_example(row):
    ref, target = row.get("reference") or "", row.get("target") or ""
    if not ref or not target:
        return None

    # -- parse the packed reference field into per-paper records
    papers, order = [], {}
    for block in ref.split(_BLOCK_SEP):
        parts = [p.strip() for p in block.split(_FIELD_SEP)]
        if len(parts) < 4:
            continue
        survey_title, chapter, abstract, bib = parts[0], parts[1], parts[2], parts[3]
        m = _BIB_RE.search(bib)
        if not m or not abstract:
            continue
        if bib not in order:
            order[bib] = len(order) + 1          # BIBnnn -> 1-based [n]
        papers.append({"n": order[bib], "abstract": abstract[:1200]})
    if len(papers) < 3:
        return None

    # -- renumber the target's markers to match
    def sub(m):
        key = f"BIB{m.group(1)}"
        return f"[{order[key]}]" if key in order else ""
    body = _BIB_RE.sub(sub, target).strip()
    if len(body) < 400:
        return None

    evidence = "\n".join(f"[{p['n']}] {p['abstract']}" for p in sorted(papers, key=lambda x: x["n"]))
    prompt = (
        f"You are writing one section of a literature survey titled "
        f'"{survey_title}", on the theme "{chapter}".\n\n'
        "Write an evidence-first thematic synthesis of the sources below. "
        "Every substantive statement must carry a bracketed citation [n] "
        "matching a source. Do not cite a source that does not support the "
        "statement, and do not write a statement you cannot cite.\n\n"
        f"SOURCES\n{evidence}\n\nSECTION:"
    )
    return {"prompt": prompt, "completion": body}

rows, seen = [], 0
for row in raw:
    seen += 1
    ex = build_example(row)
    if ex:
        rows.append(ex)
    if len(rows) >= N_TRAIN + N_EVAL:
        break

print(f"scanned {seen} rows -> kept {len(rows)}")
train_rows, eval_rows = rows[:N_TRAIN], rows[N_TRAIN:N_TRAIN + N_EVAL]
print(f"train {len(train_rows)}  eval {len(eval_rows)}")
print("\n--- sample prompt head ---")
print(train_rows[0]["prompt"][:600])

## 4 · Format as chat

In [ ]:
from datasets import Dataset

def to_chat(row):
    return {"text": tokenizer.apply_chat_template(
        [{"role": "user", "content": row["prompt"]},
         {"role": "assistant", "content": row["completion"]}],
        tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_rows).map(to_chat)
eval_ds  = Dataset.from_list(eval_rows).map(to_chat)

lens = sorted(len(tokenizer(t)["input_ids"]) for t in train_ds["text"][:200])
print("token length p50/p90/p99:", lens[len(lens)//2], lens[int(len(lens)*.9)], lens[int(len(lens)*.99)])
over = sum(1 for L in lens if L > MAX_SEQ_LEN)
print(f"{over}/{len(lens)} samples exceed MAX_SEQ_LEN={MAX_SEQ_LEN} and will truncate")
print("  (truncation cuts the END - the tail of the written section - which is")
print("   acceptable here: it teaches the opening and middle, not a stop token.)")

## 5 · Load Qwen in 4-bit and attach LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    # fp16, not bf16 - the T4 is sm_75 and has no bf16 path. Using bf16 here
    # silently falls back to fp32 accumulation and doubles the step time.
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Causal LM + right padding corrupts the loss mask on the last token. Qwen's
# own chat template assumes left padding for generation.
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False           # incompatible with gradient checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # All seven projections. Attention-only (q,v) is the common shortcut and it
    # underfits format-learning tasks like this one, where the MLP carries most
    # of the "what shape is the output" signal.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6 · Train

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=1,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                     eval_dataset=eval_ds, processing_class=tokenizer)
trainer.train()
trainer.save_model(ADAPTER_DIR)

## 7 · Evaluate — citation discipline, not fluency

ROUGE against a human chapter is the obvious metric and the wrong one: this
project is graded on **grounding**, and a fluent paragraph that cites nothing is
worse than an awkward one that cites correctly.

Three measurable properties, all cheap:

| metric | what it catches |
|---|---|
| **citation density** | claims per `[n]`. The FActScore finding is that citation presence carries almost no information — but *absence* certainly does |
| **out-of-range citations** | `[9]` when only 6 sources were given. A hallucinated reference index, and the exact defect the Verifier exists to catch downstream |
| **source coverage** | fraction of supplied sources actually cited. A writer that cites only source [1] has not synthesised anything |

In [ ]:
import torch, re
from statistics import mean

model.config.use_cache = True
model.eval()
_CIT = re.compile(r"\[(\d+)\]")

def generate(prompt, max_new_tokens=640):
    ids = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
        add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)

dens, oor, cov = [], [], []
for row in eval_rows[:40]:
    n_sources = len(set(_CIT.findall(row["prompt"])))
    text = generate(row["prompt"])
    cites = [int(x) for x in _CIT.findall(text)]
    sentences = max(1, text.count(". "))
    dens.append(len(cites) / sentences)
    oor.append(sum(1 for c in cites if c < 1 or c > n_sources))
    cov.append(len(set(cites) & set(range(1, n_sources + 1))) / max(1, n_sources))

print(f"  citations per sentence   {mean(dens):.2f}")
print(f"  out-of-range citations   {sum(oor)} across 40 sections"
      + ("   <- FABRICATED INDICES" if sum(oor) else "   <- clean"))
print(f"  source coverage          {mean(cov):.0%}")

## 8 · Adoption gate

**Adopt as the local Writer arm only if:**

1. `out-of-range citations == 0` — a citation index the reference list cannot
   resolve is fabrication, full stop
2. `citations per sentence >= 0.5`
3. `source coverage >= 0.60`
4. the report's thesis sentence is rewritten first — see the header

Expected outcome, stated in advance so it cannot be rationalised afterwards:
**SciReviewGen's own fine-tuned generators reached ~30% chapter quality.** A 3B
QLoRA on 4,000 examples matching that would be a good result. Not beating the
hosted Writer is the null hypothesis, not a failure.

In [ ]:
gate = {
    "no_out_of_range":        sum(oor) == 0,
    "citation_density_>=0.5": mean(dens) >= 0.5,
    "source_coverage_>=0.60": mean(cov) >= 0.60,
}
for k, v in gate.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print("\nVERDICT:", "ADOPT as a local arm" if all(gate.values()) else "REJECT - hosted Writer stands")

## 9 · Export for Ollama

Training happened in 4-bit on a T4. **Inference happens on a 5 GB card through
Ollama**, which wants GGUF. Three steps: merge the adapter into fp16 weights,
convert to GGUF, quantise to `q4_K_M`.

`q4_K_M` is the target because it is what the VRAM table in `agents/README.md`
is sized against - a 3B at `q4_K_M` is ~1.9 GB of weights, leaving room for an
8192-token KV cache on a 5 GB card. `q8_0` would be better quality and would not
fit alongside the context window.

⚠️ **Merging must happen in fp16, not 4-bit.** `merge_and_unload()` on a 4-bit
model silently produces garbage weights - the dequantise-merge-requantise round
trip is not lossless and the adapter deltas are small enough to be destroyed by
it. Reload the base in fp16 first, which is why this cell is slow.

In [ ]:
import torch, gc
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Free the 4-bit training copy before loading the fp16 one - a T4 will not hold
# both.
try:
    del trainer, model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

base_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"": "cpu"},
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base_fp16, "/kaggle/working/writer-qwen3b").merge_and_unload()
merged.save_pretrained("/kaggle/working/writer-qwen3b-merged", safe_serialization=True)
tokenizer.save_pretrained("/kaggle/working/writer-qwen3b-merged")
print("merged fp16 weights ->", "/kaggle/working/writer-qwen3b-merged")

In [ ]:
# llama.cpp's converter. Kaggle has internet on, so clone it here rather than
# adding a dataset dependency.
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp 2>/dev/null || true
!pip install -q -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /kaggle/working/writer-qwen3b-merged \
    --outfile /kaggle/working/researchgpt-writer-3b-f16.gguf \
    --outtype f16

# Quantise. The binary needs building once; ~3 min on Kaggle's CPUs.
!cmake -S /kaggle/working/llama.cpp -B /kaggle/working/llama.cpp/build -DLLAMA_CURL=OFF > /dev/null
!cmake --build /kaggle/working/llama.cpp/build --target llama-quantize -j4 > /dev/null

!/kaggle/working/llama.cpp/build/bin/llama-quantize \
    /kaggle/working/researchgpt-writer-3b-f16.gguf \
    /kaggle/working/researchgpt-writer-3b-q4_k_m.gguf \
    Q4_K_M

!ls -lh /kaggle/working/*.gguf

### Loading it on the 5 GB machine

Download `researchgpt-writer-3b-q4_k_m.gguf` from the Kaggle output, put it next to a
`Modelfile`, and register it with Ollama:

```
FROM ./researchgpt-writer-3b-q4_k_m.gguf

# 8192, not Ollama's default 2048. See agents/README.md - at the default,
# Ollama silently drops the front of the prompt and answers from the remainder.
PARAMETER num_ctx 8192
PARAMETER temperature 0
# The writer prompt supplies its own instructions.
```

```bash
ollama create researchgpt-writer:3b -f Modelfile
python -m agents.probe --model researchgpt-writer:3b
```

Then the scripts in `agents/` can score it against the hosted arm with no
further change.